# IMUSA V2 — Multi-Account Colab Worker 3 (Fold 4, Calibration & Ensemble)
This notebook runs **Fold 4**, gathers all 5 fold checkpoints, performs **Nelder-Mead Post-Hoc Threshold Calibration**, and evaluates the **Full 5-Fold Probability Ensemble**.

In [ ]:
# 1. Environment & GPU Setup
!nvidia-smi
!pip install -q uv
import os
import sys

if not os.path.exists("imusa-multimodal-sentiment"):
    !git clone https://github.com/shubhojit-mitra-dev/imusa-multimodal-sentiment.git
%cd /content/imusa-multimodal-sentiment
!git pull origin main
!pip install -e libs/imusa
sys.path.insert(0, "/content/imusa-multimodal-sentiment/libs/imusa/src")

In [ ]:
# 2. Google Drive Integration & Automatic data.zip Handling
import os
import shutil

from google.colab import drive, files

drive.mount("/content/drive", force_remount=False)
gdrive_zip = "/content/drive/MyDrive/data.zip"
local_zip = "/content/imusa-multimodal-sentiment/data.zip"

if os.path.exists(gdrive_zip):
    print("Found data.zip in Google Drive. Copying locally...")
    shutil.copy(gdrive_zip, local_zip)
elif not os.path.exists(local_zip):
    print("data.zip not found in Google Drive (MyDrive/data.zip).")
    print("Please select and upload data.zip from your computer now:")
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith(".zip"):
            shutil.move(fname, local_zip)
            break

# Copy to Google Drive for future runs
if os.path.exists(local_zip) and not os.path.exists(gdrive_zip):
    print("Saving data.zip to Google Drive (MyDrive/data.zip) for future runs...")
    try:
        shutil.copy(local_zip, gdrive_zip)
        print("Saved to Google Drive.")
    except Exception as e:
        print(f"Note: Could not copy to Drive: {e}")

# Unzip dataset
!unzip -q -o /content/imusa-multimodal-sentiment/data.zip -d /content/imusa-multimodal-sentiment/data/
print("Dataset extracted to data/.")

In [ ]:
# 3. Run Fold 4 Training
!python scripts/train_kfold.py --fold 4 --num-folds 5 --epochs 10 --lp-epochs 3

In [ ]:
# 4. Import Fold 0-3 Checkpoints and OOF output files from Account 1 & Account 2
# Checks Google Drive first, or prompts for upload
for fname in ["fold_0_1_outputs.zip", "fold_2_3_outputs.zip"]:
    gdrive_file = f"/content/drive/MyDrive/{fname}"
    local_file = f"/content/imusa-multimodal-sentiment/{fname}"
    if os.path.exists(gdrive_file):
        print(f"Found {fname} in Google Drive. Copying...")
        shutil.copy(gdrive_file, local_file)
    elif not os.path.exists(local_file):
        print(f"Please upload {fname}:")
        uploaded = files.upload()
        for u_name in uploaded.keys():
            if u_name == fname or u_name.endswith(".zip"):
                shutil.move(u_name, local_file)
                break

!unzip -o fold_0_1_outputs.zip
!unzip -o fold_2_3_outputs.zip
print("All 5 fold outputs ready.")

In [ ]:
# 5. Run Post-Hoc Threshold Calibration across all 5 OOF validation files
!python scripts/train_kfold.py --calibrate

In [ ]:
# 6. Generate Calibrated Ensemble Test Set Predictions
!python scripts/predict.py --model-version v2 --use-ensemble --use-calibration